# EvolveAI — Colab Control Room
This notebook launches the GitHub implementation. It validates the pipeline, checks GPU availability, and runs a bounded synthetic multi-stream test before real sources are enabled.

**Important:** The current SNN objective is byte-level next-byte prediction. This is not yet semantic reasoning.

In [ ]:
import os, sys, subprocess, torch
REPO = "/content/EvolveAI"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "https://github.com/swasthik-s/EvolveAI.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
print("Repository:", REPO)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Compute capability:", torch.cuda.get_device_capability(0))
else:
    print("CPU runtime detected. Select Runtime > Change runtime type > T4 GPU before model training.")

In [ ]:
# Run the new subsystem tests without relying on paid GitHub Actions.
tests = [
    "test_experience_evaluator.py",
    "test_parallel_data_engine.py",
    "test_online_snn_learner.py",
    "test_integrated_system.py",
]
for pattern in tests:
    print("\\n===", pattern, "===")
    subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-p", pattern, "-v"], check=True)

## Synthetic integration test
This creates 100 independent simulated connectors. It does not download 100 datasets. The test measures bounded queue behavior and confirms that selected experiences reach the SNN update path.

In [ ]:
import asyncio, tempfile
from pathlib import Path
from evolveai.system import EvolveAISystem
from evolveai.parallel_data_engine import SourceRecord

class SyntheticConnector:
    def __init__(self, source_id, count=4):
        self.source_id, self.count = source_id, count
    async def stream(self):
        for i in range(self.count):
            yield SourceRecord(
                content=f"Independent stream {self.source_id} observation {i}: recurrent spiking neural systems process sequential information and update internal weights.",
                source_id=self.source_id,
                source_uri=f"https://synthetic.invalid/{self.source_id}/{i}",
                domain="synthetic-validation",
                license="synthetic-test-data",
            )

async def run_synthetic_test():
    state = "/content/drive/MyDrive/EvolveAI/state" if os.path.isdir("/content/drive/MyDrive") else "/content/evolveai_state"
    system = EvolveAISystem(
        state_dir=state,
        device="cuda" if torch.cuda.is_available() else "cpu",
        queue_size=32,
        evaluation_workers=2,
        batch_size=8,
        train_every=8,
        hidden_size=128,
        embedding_dim=64,
        block_size=128,
    )
    for i in range(100):
        system.register(SyntheticConnector(f"sim-{i:03d}", count=4))
    status = await system.run()
    system.save()
    return status, system.status()

engine_status, system_status = await run_synthetic_test()
print("ENGINE:", engine_status)
print("SYSTEM:", system_status)

## What to inspect
- `sources_started` and `sources_completed` should reach 100.
- `max_queue_depth` must never exceed `queue_capacity`.
- `records_persisted` confirms provenance storage.
- `learner.updates` should increase only when the evaluator chooses `learn_now` and enough records accumulate.
- Checkpoint and SQLite files persist under the selected state directory.

After this synthetic test succeeds, we add real Hugging Face, RSS, encyclopedia, scientific-paper, code and API connectors with source-specific terms and licence metadata. Do not treat synthetic success as validation of external connectors or model quality.